In [2]:
from IPython.display import clear_output

!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

clear_output()

In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
print("Length of dataset in characters", len(text))

Length of dataset in characters 1115394


In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode('hello'))
print(decode(encode('hello')))

[46, 43, 50, 50, 53]
hello


In [7]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [8]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target is: {target}")

when input is tensor([18]) the target is: 47
when input is tensor([18, 47]) the target is: 56
when input is tensor([18, 47, 56]) the target is: 57
when input is tensor([18, 47, 56, 57]) the target is: 58
when input is tensor([18, 47, 56, 57, 58]) the target is: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is: 58


In [11]:
batch_size = 4 # parallel sequences
block_size = 8 # context length

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')

In [12]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # creates a vocab_size x vocab_size lookup table
        
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # plucks out the token row from the embedding table
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
            
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx) 
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss.item())
print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
4.720653057098389

K3F.DqV
eNsnxoHyIhxgTAHpL
PzMIq JttZwf,THDhKEOWCjYgPbahQr&ITBePHQnWa?!e&m?nDYNCRNdwsjvu;Tp
dsS?X-ESs


In [13]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [14]:
batch_size = 32
for step in range(10000):
    
    # sample
    xb, yb = get_batch('train')
    
    # evaluate loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(loss.item())

2.455665349960327


In [15]:
print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=300)[0].tolist()))


Anousishy

ICind spesth youne.
Y:
Sases t d cigh wono ba, bice m apiset, menie:
SAndsabes doole.
Wa y eare fofers ives nod, han f ghte
Ank bsmacimeanqulou ldlaris,
Th s,
Tor hersoakngee 's,
Ashelarourexphet.
I:
Treritielar inor, mail; Fajoncerersor n, s t feas l'd thelllmerifou maler'e we f levathat


In [25]:
B, T, C = 4, 8, 32 
x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T))
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
# out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [26]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2657, 0.7343, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3418, 0.2198, 0.4384, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1799, 0.3060, 0.1144, 0.3998, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4928, 0.1108, 0.0988, 0.2144, 0.0832, 0.0000, 0.0000, 0.0000],
        [0.0752, 0.1350, 0.2433, 0.1336, 0.2269, 0.1860, 0.0000, 0.0000],
        [0.1074, 0.1495, 0.1284, 0.1876, 0.1965, 0.1248, 0.1057, 0.0000],
        [0.0884, 0.1256, 0.1163, 0.1712, 0.1960, 0.1222, 0.0973, 0.0829]],
       grad_fn=<SelectBackward0>)